# 05 · Pretrained multilingual models: fine-tuning experiments

One notebook, run once per experiment. Set `EXPERIMENT` to a name from `configs/experiments.yaml`,
**Save Version → Save & Run All**, then attach every run's output to notebook 06.

| run | question it answers | experiment |
|---|---|---|
| `xlsr_full` | primary model: XLS-R 300M, full fine-tuning | C (model family) |
| `mms_adapter` | MMS-1B with language adapters (~2 M trainable params) | C, G |
| `xlsr_frozen` | frozen XLS-R encoder, only CTC head trained | G |
| `xlsr_lora` | LoRA on XLS-R attention | G |
| `xlsr_full_frac10/25/50` | how WER scales with the number of training speakers | A |
| `xlsr_full_minimal_text` | does transcript normalisation help or hurt? | B |
| `xlsr_full_random_split` | how much does speaker leakage inflate results? | F |
| `xlsr_full_seed2/3` | run-to-run variation of the primary model | variance |

**Suggested order under the ~30 GPU-h/week Kaggle quota:** `xlsr_full` → `mms_adapter` → `xlsr_frozen`
→ `xlsr_lora` → `frac25` → `random_split` → `minimal_text` → `frac10`, `frac50` → seeds.
Read the *throughput check* line after 50 steps; if the projected time exceeds the session, lower
`train.max_steps` for **all** comparable runs (record that in the report) rather than for one run.

Before choosing XLS-R vs MMS checkpoints: neither model card is assumed to cover Dholuo. XLS-R is a
self-supervised encoder without any ASR head; MMS's language coverage is checked in notebook 03.

In [ ]:
# ── Setup: locate the project source (src/dholuo_asr) and the output folder ──────────
import os, sys, subprocess, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# After pushing the repo to GitHub, put its URL here (Kaggle: Settings → Internet ON).
# Alternatively upload the repo as a private Kaggle dataset and attach it as an input.
REPO_URL = os.environ.get("DHOLUO_REPO_URL", "https://github.com/YOUR-USERNAME/dholuo-asr.git")

def _find_src(max_depth=4):
    for root in [Path.cwd(), Path.cwd().parent, Path("/kaggle/working"), Path("/kaggle/input")]:
        if not root.exists():
            continue
        base = len(root.parts)
        for dirpath, dirnames, filenames in os.walk(root):
            if len(Path(dirpath).parts) - base >= max_depth or len(filenames) > 500:
                dirnames[:] = []          # don't descend into audio folders
            if Path(dirpath).name == "dholuo_asr" and "__init__.py" in filenames:
                return Path(dirpath).parent
    return None

SRC = _find_src()
if SRC is None and "YOUR-USERNAME" not in REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/kaggle/working/dholuo-asr"], check=True)
    SRC = Path("/kaggle/working/dholuo-asr/src")
if SRC is None:
    raise RuntimeError("Project source not found: set REPO_URL (Internet ON) or attach the repo as a dataset.")
sys.path.insert(0, str(SRC))
REPO = SRC.parent
ON_KAGGLE = Path("/kaggle/working").exists()
OUT = Path(os.environ.get("DHOLUO_OUT", "/kaggle/working" if ON_KAGGLE else REPO / "outputs"))
OUT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOTS = [OUT, Path("/kaggle/input")]
print("source :", SRC)
print("outputs:", OUT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})
DIALECT_COLORS = {"Milambo": "#2a6f97", "Nyanduat": "#e07a5f"}
FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG / f"{name}.png", bbox_inches="tight"); plt.show()

In [ ]:
# ← set this before each run
EXPERIMENT = os.environ.get("EXPERIMENT", "xlsr_full")
OVERRIDES = {}       # one-off overrides, e.g. {"train.max_minutes": 500}; prefer editing the YAML
try:
    import peft  # noqa: F401  (needed for xlsr_lora only)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=False)

In [ ]:
import torch
from dholuo_asr.utils import find_artifact, load_experiment, dump_json
from dholuo_asr.runner import run_experiment
CONFIG = REPO / "configs" / "experiments.yaml"
cfg = load_experiment(CONFIG, EXPERIMENT)
for k, v in OVERRIDES.items():          # e.g. {"train.max_steps": 3000}
    sect, key = k.split(".")
    cfg[sect][key] = v
print(json.dumps(cfg, indent=1))
splits = pd.read_csv(find_artifact("splits.csv", ARTIFACT_ROOTS), keep_default_na=False, na_values=[""])
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
metrics = run_experiment(cfg, splits, OUT / "results", log=print)

### Training curve (validation WER is the model-selection criterion; the test set is never consulted)

In [ ]:
RUN = OUT / "results" / cfg["name"]
log = pd.read_csv(RUN / "train_log.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(log["step"], log["train_loss"], marker="o"); ax[0].set_title("training CTC loss"); ax[0].set_xlabel("step")
ax[1].plot(log["step"], log["val_wer"], marker="o", label="val WER"); ax[1].plot(log["step"], log["val_cer"], marker="o", label="val CER")
ax[1].axvline(metrics["best"]["step"], ls="--", c="k", lw=1, label="selected checkpoint"); ax[1].legend(); ax[1].set_xlabel("step")
savefig(f"curve_{cfg['name']}")
summary = {s: {k: metrics[s][k] for k in ["wer", "cer", "wer_ci", "cer_ci", "n_utts"]} for s in ["val", "test"]}
print(json.dumps({"best": metrics["best"], "params_trainable": metrics["params_trainable"], **summary}, indent=1))
pd.read_csv(RUN / "predictions_test.csv")[["dialect", "ref", "hyp"]].sample(10, random_state=0)